In [1]:
import pandas as pd, numpy as np
import sys, os
from pathlib import Path

# Ensure we are at the project root (the folder that contains `src/`)
# If your notebook sits in the root, this is already correct.
root = Path.cwd()

# If your notebook lives somewhere else, climb up until we see 'src'
while not (root / "src").exists() and root.parent != root:
    root = root.parent

# Put project root and src/ on sys.path
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "src"))

print("Project root:", root)
print("Has src?:", (root / "src").exists())

# Force a clean import of the latest file
import importlib, src.esm_feats as esm_feats
importlib.reload(esm_feats)

# See what the module actually exports
print([n for n in dir(esm_feats) if "emb" in n or "shot" in n or "load" in n])

from src.esm_feats import embed_dataframe, load_esm1v


Project root: d:\Bioinformatics\Rosaloid
Has src?: True
['__loader__', 'embed_dataframe', 'get_embedding', 'load_esm1v', 'zero_shot_dataframe', 'zero_shot_dataframe_additive_fast', 'zero_shot_dataframe_mutantctx_batched', 'zero_shot_score']


In [4]:
# round0.py — minimal, no-embedding selector for Round-0
# Produces:
#   - round0_additive_top.csv
#   - round0_plldelta_top.csv
#   - round0_mutantctx_top.csv
#   - round0_union_panel.csv
#
# Expected input CSVs (under IN_DIR):
#   - gfp_dms_with_additive_zeroshot.csv   (score column: esm1v_zero_shot_add)
#   - gfp_dms_with_plldelta.csv            (score column: pll_delta)
#   - gfp_dms_with_zeroshot_mutantctx.csv  (score column: esm1v_zero_shot_mc)

from pathlib import Path
import pandas as pd
import numpy as np

# ---------- config ----------
IN_DIR = (Path.cwd().parent / "zeroshot_prior").resolve()
OUT_DIR = Path(".").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_PER_METHOD   = 96        # top-k per method
MAX_SUBS       = 2         # trust radius
REQUIRE_SUBS   = False     # set True to hard-fail if num_subs missing
WT_SEQ         = None      # optional: set to WT AA string to compute num_subs if missing

SCHEMAS = {
    "additive":  dict(path=IN_DIR/"gfp_dms_with_additive_zeroshot.csv",
                      key="mutated_sequence", score="esm1v_zero_shot_add",
                      subs="num_subs", label="additive",
                      out=OUT_DIR/"round0_additive_top.csv"),
    "plldelta":  dict(path=IN_DIR/"gfp_dms_with_plldelta.csv",
                      key="mutated_sequence", score="pll_delta",
                      subs="num_subs", label="plldelta",
                      out=OUT_DIR/"round0_plldelta_top.csv"),
    "mutantctx": dict(path=IN_DIR/"gfp_dms_with_zeroshot_mutantctx.csv",
                      key="mutated_sequence", score="esm1v_zero_shot_mc",
                      subs="num_subs", label="mutant_ctx",
                      out=OUT_DIR/"round0_mutantctx_top.csv"),
}

def _require_cols(df, cols, where):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise ValueError(f"{where}: missing {miss}. Have: {list(df.columns)}")

def _ensure_num_subs(df, subs_col, key_col):
    df = df.copy()
    if subs_col in df.columns:
        df["num_subs"] = pd.to_numeric(df[subs_col], errors="coerce")
        return df
    if WT_SEQ is not None and key_col in df.columns:
        if df[key_col].map(len).eq(len(WT_SEQ)).all():
            df["num_subs"] = df[key_col].map(lambda s: sum(a != b for a,b in zip(s, WT_SEQ)))
            return df
    if REQUIRE_SUBS:
        raise ValueError("num_subs missing and WT_SEQ not set; set WT_SEQ or REQUIRE_SUBS=False.")
    df["num_subs"] = np.nan
    return df

def _prep(schema):
    p,key,score,subs,label = schema["path"],schema["key"],schema["score"],schema["subs"],schema["label"]
    if not p.exists():
        raise FileNotFoundError(f"[{label}] missing file: {p}")
    df = pd.read_csv(p)
    _require_cols(df, [key, score], f"[{label}] {p.name}")
    df = _ensure_num_subs(df, subs, key)
    out = df[[key, "num_subs", score]].copy()
    out.columns = ["mutated_sequence","num_subs","score"]
    out["method"] = label
    # strong typing
    out["mutated_sequence"] = out["mutated_sequence"].astype(str)
    out["score"] = pd.to_numeric(out["score"], errors="coerce")
    return out

def _filter_trust(df, max_subs):
    if df["num_subs"].isna().all():
        print("WARNING: num_subs missing; skipping trust-radius filter.")
        return df
    return df.loc[df["num_subs"] <= max_subs].copy()

def _top_unique(df, n):
    ranked = df.sort_values("score", ascending=False)
    uniq   = ranked.drop_duplicates(subset=["mutated_sequence"], keep="first")
    return uniq.head(n).reset_index(drop=True)

def _brief(df, name):
    if len(df) == 0:
        print(f"{name}: 0 rows")
        return
    smin = np.nanmin(df["score"].values); smax = np.nanmax(df["score"].values)
    known = int((~df["num_subs"].isna()).sum())
    print(f"{name}: {len(df):4d} rows | score [{smin:.3g}, {smax:.3g}] | num_subs known {known}/{len(df)}")

# ---------- run ----------
add_df  = _prep(SCHEMAS["additive"])
pll_df  = _prep(SCHEMAS["plldelta"])
ctx_df  = _prep(SCHEMAS["mutantctx"])

add_df = _filter_trust(add_df, MAX_SUBS)
pll_df = _filter_trust(pll_df, MAX_SUBS)
ctx_df = _filter_trust(ctx_df, MAX_SUBS)

add_top = _top_unique(add_df, N_PER_METHOD);  add_top.to_csv(SCHEMAS["additive"]["out"], index=False)
pll_top = _top_unique(pll_df, N_PER_METHOD);  pll_top.to_csv(SCHEMAS["plldelta"]["out"], index=False)
ctx_top = _top_unique(ctx_df, N_PER_METHOD);  ctx_top.to_csv(SCHEMAS["mutantctx"]["out"], index=False)

union_panel = pd.concat([add_top, pll_top, ctx_top], ignore_index=True)
union_panel = (union_panel
               .sort_values(["mutated_sequence","score"], ascending=[True,False])
               .drop_duplicates(subset=["mutated_sequence"], keep="first")
               .reset_index(drop=True))
union_panel.to_csv(OUT_DIR/"round0_union_panel.csv", index=False)

print("\n=== pools (after trust filter) ===")
_brief(add_df, "additive pool")
_brief(pll_df, "plldelta pool")
_brief(ctx_df, "mutant_ctx pool")

print("\n=== per-method top-96 ===")
_brief(add_top, "additive top")
_brief(pll_top, "plldelta top")
_brief(ctx_top, "mutant_ctx top")

overlap = {
    "add∩pll": len(set(add_top.mutated_sequence)&set(pll_top.mutated_sequence)),
    "add∩ctx": len(set(add_top.mutated_sequence)&set(ctx_top.mutated_sequence)),
    "pll∩ctx": len(set(pll_top.mutated_sequence)&set(ctx_top.mutated_sequence)),
}
tri = set(add_top.mutated_sequence) & set(pll_top.mutated_sequence) & set(ctx_top.mutated_sequence)
print(f"\nOverlap: {overlap} | 3-way: {len(tri)}")
print("\nWrote:\n- round0_additive_top.csv\n- round0_plldelta_top.csv\n- round0_mutantctx_top.csv\n- round0_union_panel.csv")



=== pools (after trust filter) ===
additive pool: 13861 rows | score [-4.92, 3.81] | num_subs known 13861/13861
plldelta pool: 13861 rows | score [-11.7, 6.52] | num_subs known 13861/13861
mutant_ctx pool: 13861 rows | score [-4.83, 3.85] | num_subs known 13861/13861

=== per-method top-96 ===
additive top:   96 rows | score [2.27, 3.81] | num_subs known 96/96
plldelta top:   96 rows | score [4, 6.52] | num_subs known 96/96
mutant_ctx top:   96 rows | score [2.32, 3.85] | num_subs known 96/96

Overlap: {'add∩pll': 24, 'add∩ctx': 90, 'pll∩ctx': 22} | 3-way: 22

Wrote:
- round0_additive_top.csv
- round0_plldelta_top.csv
- round0_mutantctx_top.csv
- round0_union_panel.csv
